
# KMZ / KML → ASCII PLY Waypoints (with edges) + `relativeToStartPoint` height handling

This notebook is an improved version of the KMZ/KML waypoint converter.

## What it does
- accepts a single **`.kmz` / `.zip` / `.kml`** or a **folder** (recursive)
- extracts `template.kml` from KMZ/ZIP into `/KML`
- parses DJI route waypoints from `<Placemark><Point><coordinates>...</coordinates>`
- converts to a target CRS (**default: Lambert 2008 / EPSG:3812**)
- exports **ASCII PLY** with:
  - **vertices**
  - **edges** connecting points in waypoint order (Meshlab-friendly)
- optional TXT sidecar export (waypoint table)

## New improvements
1. **`<wpml:heightMode>relativeToStartPoint</wpml:heightMode>` support**
   - if a route uses `relativeToStartPoint`, the notebook computes actual ellipsoidal height as:
     **`z_ellipsoidal = start_height + wpml:ellipsoidHeight`**
   - in batch mode, you can either:
     - predefine a value per route in a dictionary, or
     - let the notebook **prompt in the CLI** (via `input()`) for each route

2. **PLY edges are written explicitly**
   - header includes `element edge ...`
   - edges are written as `vertex1 vertex2` pairs (`0 1`, `1 2`, ...)

## Belgian correction grids
Default grid folder (Windows):
`C:\Code\_Orbitv1.0.4\orbit\tools\KMZ2OBJ`

The folder may contain:
- `_BD72LB72_ETRS89LB08.gsb`
- `hBG18.geo`

These are added to PROJ's local data search path if present.


In [ ]:

# If needed (run once):
# !pip install pyproj pandas tqdm ipywidgets

import os
import zipfile
import warnings
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

from pyproj import CRS
from pyproj import datadir as pyproj_datadir
from pyproj.transformer import TransformerGroup

print("Imports OK")


#### Config (run all, close)

In [ ]:

# =========================
# User configuration
# =========================

INPUT_PATH = r""  # file or folder

TARGET_CRS = "lambert2008"
HEIGHT_MODE = "ellipsoidal"
GRID_DIR = r"C:\Code\_Orbitv1.0.4\orbit\tools\KMZ2OBJ"

SORT_BY_WPML_INDEX = False
OVERWRITE = True

# Distance-based sampling after transformation (meters)
# Keeps first point, then keeps the next point whenever cumulative distance
# reaches/exceeds SAMPLE_SPACING_M, and always keeps the last point.
RESAMPLE_BY_DISTANCE = True
SAMPLE_SPACING_M = 0.005  # meters

# relativeToStartPoint support
START_HEIGHTS_BY_ROUTE = {
    # "RouteName": 52.7421875,
}
DEFAULT_START_ELLIPSOID_HEIGHT = None
PROMPT_START_HEIGHT_PER_ROUTE = True

# Optional TXT sidecar
EXPORT_TXT = False
TXT_DELIMITER = "\t"


In [ ]:

# =========================
# Core functions
# =========================

import math

def _configure_proj_grids(grid_dir: Optional[str]) -> bool:
    if not grid_dir:
        return False
    p = Path(grid_dir)
    if not p.exists():
        warnings.warn(f"Grid directory does not exist: {p}")
        return False
    try:
        pyproj_datadir.append_data_dir(str(p))
    except Exception:
        pass
    os.environ.setdefault("PROJ_NETWORK", "OFF")
    return True


def _resolve_target_crs(target: str) -> CRS:
    t = str(target).strip().lower()
    if t in {"lambert2008", "lambert 2008", "lb08", "3812", "epsg:3812"}:
        return CRS.from_epsg(3812)
    if t in {"lambert72", "lambert 72", "lb72", "31370", "epsg:31370"}:
        return CRS.from_epsg(31370)
    return CRS.from_user_input(target)


def _get_transformer(target: str, grid_dir: Optional[str] = None):
    _configure_proj_grids(grid_dir)
    src_crs = CRS.from_epsg(4979)  # WGS84 3D
    dst_crs = _resolve_target_crs(target)
    tg = TransformerGroup(src_crs, dst_crs, always_xy=True)
    if not tg.transformers:
        raise RuntimeError(f"No transformation available from {src_crs} to {dst_crs}")
    return tg.transformers[0], src_crs, dst_crs, tg


def _collect_input_files(input_path: Path) -> List[Path]:
    if not input_path.exists():
        raise FileNotFoundError(f"Input path does not exist: {input_path}")
    if input_path.is_file():
        if input_path.suffix.lower() not in {".kmz", ".zip", ".kml"}:
            raise ValueError(f"Unsupported file type: {input_path.suffix}")
        return [input_path]
    return sorted([p for p in input_path.rglob("*") if p.is_file() and p.suffix.lower() in {".kmz", ".zip", ".kml"}])


def _safe_relative(path: Path, root: Path) -> Path:
    try:
        return path.relative_to(root)
    except Exception:
        return Path(path.name)


def _extract_template_kml_from_zip(zip_path: Path, kml_dir: Path, rel_root: Path, overwrite: bool = True) -> Path:
    kml_dir.mkdir(parents=True, exist_ok=True)
    rel = _safe_relative(zip_path, rel_root)
    out_subdir = kml_dir / rel.parent
    out_subdir.mkdir(parents=True, exist_ok=True)
    out_kml = out_subdir / f"{zip_path.stem}__template.kml"
    if out_kml.exists() and not overwrite:
        return out_kml

    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()
        candidates = [n for n in names if n.lower().endswith("template.kml")]
        if not candidates:
            kml_files = [n for n in names if n.lower().endswith(".kml")]
            if len(kml_files) == 1:
                candidates = kml_files
        if not candidates:
            raise FileNotFoundError(f"No template.kml (or unique .kml) found inside: {zip_path}")
        data = zf.read(candidates[0])
        out_kml.write_bytes(data)

    return out_kml


def _find_text(el, xpath: str) -> Optional[str]:
    node = el.find(xpath)
    if node is None or node.text is None:
        return None
    txt = node.text.strip()
    return txt if txt else None


def _parse_waypoints_from_kml(kml_path: Path, sort_by_wpml_index: bool = False) -> Tuple[List[Dict[str, Any]], Optional[str]]:
    tree = ET.parse(kml_path)
    root = tree.getroot()

    route_height_mode = None
    for hm in root.findall(".//{*}heightMode"):
        txt = (hm.text or "").strip()
        if txt:
            route_height_mode = txt
            break

    waypoints = []
    occurrence = 0

    for pm in root.findall(".//{*}Placemark"):
        coord_txt = _find_text(pm, ".//{*}Point/{*}coordinates")
        if not coord_txt:
            continue

        parts = [p.strip() for p in coord_txt.split(",")]
        if len(parts) < 2:
            continue

        try:
            lon = float(parts[0]); lat = float(parts[1])
        except ValueError:
            continue

        coord_alt = None
        if len(parts) >= 3 and parts[2] != "":
            try:
                coord_alt = float(parts[2])
            except ValueError:
                coord_alt = None

        idx_txt = _find_text(pm, ".//{*}index")
        wpml_index = None
        if idx_txt is not None:
            try:
                wpml_index = int(float(idx_txt))
            except ValueError:
                wpml_index = None

        eh_txt = _find_text(pm, ".//{*}ellipsoidHeight")
        ellipsoid_height = None
        if eh_txt is not None:
            try:
                ellipsoid_height = float(eh_txt)
            except ValueError:
                ellipsoid_height = None

        point_height_mode = _find_text(pm, ".//{*}heightMode")
        if point_height_mode is None:
            point_height_mode = route_height_mode

        if ellipsoid_height is None:
            ellipsoid_height = coord_alt

        waypoints.append({
            "occurrence": occurrence,
            "wpml_index": wpml_index,
            "lon": lon,
            "lat": lat,
            "ellipsoid_height_raw": ellipsoid_height,
            "coord_alt": coord_alt,
            "height_mode": point_height_mode,
        })
        occurrence += 1

    if sort_by_wpml_index:
        waypoints.sort(key=lambda r: (r["wpml_index"] is None, r["wpml_index"], r["occurrence"]))

    return waypoints, route_height_mode


def _resolve_start_height_for_route(
    source_file: Path,
    kml_path: Path,
    start_heights_by_route: Optional[Dict[str, float]] = None,
    default_start_height: Optional[float] = None,
    prompt_if_missing: bool = True,
) -> float:
    d = start_heights_by_route or {}
    keys_to_try = [source_file.stem, source_file.name, kml_path.stem, kml_path.name]
    for key in keys_to_try:
        if key in d:
            return float(d[key])

    if default_start_height is not None:
        return float(default_start_height)

    if prompt_if_missing:
        while True:
            raw = input(
                f"Route '{source_file.name}' uses heightMode=relativeToStartPoint. "
                f"Enter start ellipsoidal height [m]: "
            ).strip()
            try:
                return float(raw)
            except ValueError:
                print("Please enter a numeric value (e.g. 52.7421875).")

    raise ValueError(
        f"Route '{source_file.name}' uses relativeToStartPoint but no start height was provided."
    )


def _compute_actual_ellipsoidal_heights(
    waypoints: List[Dict[str, Any]],
    source_file: Path,
    kml_path: Path,
    start_heights_by_route: Optional[Dict[str, float]] = None,
    default_start_height: Optional[float] = None,
    prompt_if_missing: bool = True,
) -> Tuple[List[Dict[str, Any]], Optional[float], bool]:
    requires_start = any(
        str(wp.get("height_mode") or "").strip().lower() == "relativetostartpoint"
        for wp in waypoints
    )

    start_height_used = None
    if requires_start:
        start_height_used = _resolve_start_height_for_route(
            source_file=source_file,
            kml_path=kml_path,
            start_heights_by_route=start_heights_by_route,
            default_start_height=default_start_height,
            prompt_if_missing=prompt_if_missing,
        )

    out = []
    for wp in waypoints:
        wp2 = dict(wp)
        raw_h = wp2.get("ellipsoid_height_raw")
        if raw_h is None:
            raw_h = 0.0

        hm = str(wp2.get("height_mode") or "").strip().lower()
        if hm == "relativetostartpoint":
            actual_h = float(start_height_used) + float(raw_h)
        else:
            actual_h = float(raw_h)

        wp2["ellipsoid_height_actual"] = actual_h
        out.append(wp2)

    return out, start_height_used, requires_start


def _transform_waypoints(
    waypoints: List[Dict[str, Any]],
    target_crs: str = "lambert2008",
    height_mode: str = "ellipsoidal",
    grid_dir: Optional[str] = None,
):
    transformer, src_crs, dst_crs, tg = _get_transformer(target_crs, grid_dir)

    if str(height_mode).lower() != "ellipsoidal":
        warnings.warn("This notebook currently outputs ellipsoidal heights by default.")

    xyz = []
    for wp in waypoints:
        h = wp.get("ellipsoid_height_actual")
        if h is None:
            h = 0.0

        res = transformer.transform(float(wp["lon"]), float(wp["lat"]), float(h))
        if isinstance(res, tuple) and len(res) >= 3:
            x, y, z = res[0], res[1], res[2]
        elif isinstance(res, tuple) and len(res) == 2:
            x, y = res
            z = h
        else:
            raise RuntimeError("Unexpected transformer output.")
        xyz.append((float(x), float(y), float(z)))

    return xyz, src_crs, dst_crs, tg


def _sample_waypoints_by_distance(
    waypoints: List[Dict[str, Any]],
    xyz: List[Tuple[float, float, float]],
    enabled: bool = True,
    spacing_m: float = 0.005,
) -> Tuple[List[Dict[str, Any]], List[Tuple[float, float, float]]]:
    """Distance-based thinning along the polyline after CRS conversion."""
    if not enabled:
        return waypoints, xyz
    if spacing_m is None:
        return waypoints, xyz
    try:
        spacing_m = float(spacing_m)
    except Exception:
        raise ValueError(f"Invalid SAMPLE_SPACING_M: {spacing_m}")
    if spacing_m <= 0:
        return waypoints, xyz
    if len(xyz) <= 2:
        return waypoints, xyz

    keep_idx = [0]
    cum = 0.0

    for i in range(1, len(xyz)):
        x0, y0, z0 = xyz[i - 1]
        x1, y1, z1 = xyz[i]
        seg = math.sqrt((x1 - x0)**2 + (y1 - y0)**2 + (z1 - z0)**2)
        cum += seg
        if cum >= spacing_m:
            keep_idx.append(i)
            cum = 0.0

    if keep_idx[-1] != len(xyz) - 1:
        keep_idx.append(len(xyz) - 1)

    sampled_waypoints = [waypoints[i] for i in keep_idx]
    sampled_xyz = [xyz[i] for i in keep_idx]
    return sampled_waypoints, sampled_xyz


def _write_ply_ascii_with_edges(
    out_ply: Path,
    xyz: List[Tuple[float, float, float]],
    dst_crs: CRS,
    overwrite: bool = True,
) -> Path:
    out_ply.parent.mkdir(parents=True, exist_ok=True)
    if out_ply.exists() and not overwrite:
        return out_ply

    n_vertices = len(xyz)
    n_edges = max(0, n_vertices - 1)

    with out_ply.open("w", encoding="utf-8", newline="\n") as f:
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write("comment Generated from KML/KMZ waypoints\n")
        f.write(f"comment CRS: {dst_crs.to_string()}\n")
        f.write(f"element vertex {n_vertices}\n")
        f.write("property float x\n")
        f.write("property float y\n")
        f.write("property float z\n")
        f.write(f"element edge {n_edges}\n")
        f.write("property int vertex1\n")
        f.write("property int vertex2\n")
        f.write("end_header\n")

        for x, y, z in xyz:
            f.write(f"{x:.6f} {y:.6f} {z:.6f}\n")
        for i in range(n_edges):
            f.write(f"{i} {i+1}\n")

    return out_ply


def _write_txt_waypoints(
    out_txt: Path,
    waypoints: List[Dict[str, Any]],
    xyz: List[Tuple[float, float, float]],
    delimiter: str = "\t",
    overwrite: bool = True,
) -> Path:
    out_txt.parent.mkdir(parents=True, exist_ok=True)
    if out_txt.exists() and not overwrite:
        return out_txt

    with out_txt.open("w", encoding="utf-8", newline="\n") as f:
        f.write(delimiter.join([
            "route_point_name", "wpml_index", "lon", "lat",
            "height_mode", "ellipsoid_height_raw", "ellipsoid_height_actual",
            "x", "y", "z"
        ]) + "\n")

        for i, (wp, (x, y, z)) in enumerate(zip(waypoints, xyz)):
            f.write(delimiter.join([
                f"WP_{i:03d}",
                "" if wp.get("wpml_index") is None else str(wp.get("wpml_index")),
                f"{float(wp['lon']):.12f}",
                f"{float(wp['lat']):.12f}",
                str(wp.get("height_mode") or ""),
                "" if wp.get("ellipsoid_height_raw") is None else str(float(wp.get("ellipsoid_height_raw"))),
                "" if wp.get("ellipsoid_height_actual") is None else str(float(wp.get("ellipsoid_height_actual"))),
                f"{float(x):.6f}",
                f"{float(y):.6f}",
                f"{float(z):.6f}",
            ]) + "\n")

    return out_txt


In [ ]:

# =========================
# Main processing
# =========================

def process_kmz_kml_to_ply(
    input_path: str,
    target_crs: str = "lambert2008",
    height_mode: str = "ellipsoidal",
    grid_dir: Optional[str] = None,
    sort_by_wpml_index: bool = False,
    overwrite: bool = True,
    start_heights_by_route: Optional[Dict[str, float]] = None,
    default_start_ellipsoid_height: Optional[float] = None,
    prompt_start_height_per_route: bool = True,
    export_txt: bool = False,
    txt_delimiter: str = "\t",
    resample_by_distance: bool = True,
    sample_spacing_m: float = 0.005,
):
    input_path = Path(input_path)
    root = input_path if input_path.is_dir() else input_path.parent
    kml_dir = root / "KML"
    ply_dir = root / "PLY"
    txt_dir = root / "TXT"

    files = _collect_input_files(input_path)
    if not files:
        raise FileNotFoundError("No .kmz/.zip/.kml files found.")

    records = []
    rel_root = root

    for src in tqdm(files, desc="Processing routes", colour="yellow"):
        suffix = src.suffix.lower()

        if suffix in {".kmz", ".zip"}:
            try:
                kml_path = _extract_template_kml_from_zip(src, kml_dir, rel_root, overwrite=overwrite)
            except Exception as e:
                records.append({
                    "source_file": str(src), "kml_path": None, "ply_path": None, "txt_path": None,
                    "n_waypoints_original": 0, "n_waypoints_output": 0, "status": f"ERROR extracting KML: {e}"
                })
                continue
        elif suffix == ".kml":
            kml_path = src
        else:
            continue

        try:
            waypoints, route_height_mode = _parse_waypoints_from_kml(kml_path, sort_by_wpml_index=sort_by_wpml_index)
            if not waypoints:
                records.append({
                    "source_file": str(src), "kml_path": str(kml_path), "ply_path": None, "txt_path": None,
                    "n_waypoints_original": 0, "n_waypoints_output": 0, "status": "No waypoint placemarks found"
                })
                continue
        except Exception as e:
            records.append({
                "source_file": str(src), "kml_path": str(kml_path), "ply_path": None, "txt_path": None,
                "n_waypoints_original": 0, "n_waypoints_output": 0, "status": f"ERROR parsing KML: {e}"
            })
            continue

        try:
            waypoints_h, start_height_used, requires_start = _compute_actual_ellipsoidal_heights(
                waypoints=waypoints,
                source_file=src,
                kml_path=kml_path,
                start_heights_by_route=start_heights_by_route,
                default_start_height=default_start_ellipsoid_height,
                prompt_if_missing=prompt_start_height_per_route,
            )
        except Exception as e:
            records.append({
                "source_file": str(src), "kml_path": str(kml_path), "ply_path": None, "txt_path": None,
                "n_waypoints_original": len(waypoints), "n_waypoints_output": 0,
                "status": f"ERROR resolving heights: {e}"
            })
            continue

        try:
            xyz, src_crs_obj, dst_crs_obj, tg = _transform_waypoints(
                waypoints_h, target_crs=target_crs, height_mode=height_mode, grid_dir=grid_dir
            )
        except Exception as e:
            records.append({
                "source_file": str(src), "kml_path": str(kml_path), "ply_path": None, "txt_path": None,
                "n_waypoints_original": len(waypoints_h), "n_waypoints_output": 0,
                "status": f"ERROR transforming: {e}"
            })
            continue

        try:
            waypoints_out, xyz_out = _sample_waypoints_by_distance(
                waypoints_h, xyz, enabled=resample_by_distance, spacing_m=sample_spacing_m
            )
        except Exception as e:
            records.append({
                "source_file": str(src), "kml_path": str(kml_path), "ply_path": None, "txt_path": None,
                "n_waypoints_original": len(waypoints_h), "n_waypoints_output": 0,
                "status": f"ERROR sampling: {e}"
            })
            continue

        rel_src = _safe_relative(src, rel_root)
        out_ply = (ply_dir / rel_src).with_suffix(".ply")
        out_txt = (txt_dir / rel_src).with_suffix(".txt")

        try:
            _write_ply_ascii_with_edges(out_ply=out_ply, xyz=xyz_out, dst_crs=dst_crs_obj, overwrite=overwrite)
        except Exception as e:
            records.append({
                "source_file": str(src), "kml_path": str(kml_path), "ply_path": None, "txt_path": None,
                "n_waypoints_original": len(waypoints_h), "n_waypoints_output": len(xyz_out),
                "status": f"ERROR writing PLY: {e}"
            })
            continue

        txt_written = None
        txt_err = None
        if export_txt:
            try:
                _write_txt_waypoints(out_txt=out_txt, waypoints=waypoints_out, xyz=xyz_out,
                                     delimiter=txt_delimiter, overwrite=overwrite)
                txt_written = str(out_txt)
            except Exception as e:
                txt_err = str(e)

        status = "OK" if txt_err is None else f"PLY OK, TXT ERROR: {txt_err}"

        records.append({
            "source_file": str(src),
            "kml_path": str(kml_path),
            "ply_path": str(out_ply),
            "txt_path": txt_written,
            "n_waypoints_original": len(waypoints_h),
            "n_waypoints_output": len(waypoints_out),
            "route_height_mode_detected": route_height_mode,
            "relativeToStartPoint_used": requires_start,
            "start_ellipsoid_height_used": start_height_used,
            "sampling_enabled": bool(resample_by_distance),
            "sample_spacing_m": float(sample_spacing_m) if resample_by_distance else None,
            "target_crs": dst_crs_obj.to_string(),
            "best_transform_available": getattr(tg, "best_available", None),
            "status": status,
        })

    summary = pd.DataFrame(records) if (pd is not None and len(records) > 0) else records
    return records, summary


#### CLI (run)

In [ ]:

# =========================
# Run (set INPUT_PATH first)
# =========================

if not INPUT_PATH:
    print("Set INPUT_PATH in the config cell above (file or folder) and run again.")
else:
    records, summary = process_kmz_kml_to_ply(
        input_path=INPUT_PATH,
        target_crs=TARGET_CRS,
        height_mode=HEIGHT_MODE,
        grid_dir=GRID_DIR,
        sort_by_wpml_index=SORT_BY_WPML_INDEX,
        overwrite=OVERWRITE,
        start_heights_by_route=START_HEIGHTS_BY_ROUTE,
        default_start_ellipsoid_height=DEFAULT_START_ELLIPSOID_HEIGHT,
        prompt_start_height_per_route=PROMPT_START_HEIGHT_PER_ROUTE,
        export_txt=EXPORT_TXT,
        txt_delimiter=TXT_DELIMITER,
        resample_by_distance=RESAMPLE_BY_DISTANCE,
        sample_spacing_m=SAMPLE_SPACING_M,
    )

    root = Path(INPUT_PATH) if Path(INPUT_PATH).is_dir() else Path(INPUT_PATH).parent
    print("\nDone.")
    print(f"PLY output directory: {root / 'PLY'}")
    if EXPORT_TXT:
        print(f"TXT output directory: {root / 'TXT'}")

    if pd is not None and hasattr(summary, "head"):
        display(summary)
    else:
        for r in summary:
            print(r)


In [ ]:

# =========================
# Optional mini UI (ipywidgets) - route-wise start height fields
# =========================
# Workflow:
# 1) Click "Process" -> scans routes and lists them line-by-line
# 2) For routes using relativeToStartPoint, enter start ellipsoidal height in the text field
# 3) Click "Proceed" -> runs conversion using those values
#
# Progress bars are yellow (bar_style='warning').

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    def _fmt_float_or_blank(v):
        return "" if v is None else str(v)

    def _resolve_prefill_start_height(src_path_obj, kml_path_obj):
        try:
            d = START_HEIGHTS_BY_ROUTE if isinstance(START_HEIGHTS_BY_ROUTE, dict) else {}
        except Exception:
            d = {}
        keys = [src_path_obj.stem, src_path_obj.name, kml_path_obj.stem, kml_path_obj.name]
        for k in keys:
            if k in d:
                try:
                    return float(d[k])
                except Exception:
                    pass
        try:
            if DEFAULT_START_ELLIPSOID_HEIGHT is not None:
                return float(DEFAULT_START_ELLIPSOID_HEIGHT)
        except Exception:
            pass
        return None

    path_w = widgets.Text(
        value=INPUT_PATH if INPUT_PATH else "",
        placeholder=r"C:\path\to\route.kmz or folder",
        description="Input:",
        layout=widgets.Layout(width="95%"),
    )

    crs_w = widgets.Dropdown(
        options=["lambert2008", "lambert72"],
        value=TARGET_CRS if TARGET_CRS in ["lambert2008", "lambert72"] else "lambert2008",
        description="CRS:",
    )

    sort_w = widgets.Checkbox(value=SORT_BY_WPML_INDEX, description="Sort by wpml:index")
    txt_w = widgets.Checkbox(value=EXPORT_TXT, description="Export TXT")
    overwrite_w = widgets.Checkbox(value=OVERWRITE, description="Overwrite")

    sample_w = widgets.Checkbox(value=RESAMPLE_BY_DISTANCE, description="Sample by distance")
    sample_dist_w = widgets.Text(
        value=str(SAMPLE_SPACING_M),
        description="Spacing [m]:",
        layout=widgets.Layout(width="220px"),
    )

    process_btn = widgets.Button(description="Process", button_style="primary")
    proceed_btn = widgets.Button(description="Proceed", button_style="success", disabled=True)

    scan_progress = widgets.IntProgress(value=0, min=0, max=1, description="Scan",
                                        bar_style="warning", layout=widgets.Layout(width="420px"))
    proc_progress = widgets.IntProgress(value=0, min=0, max=1, description="Routes",
                                        bar_style="warning", layout=widgets.Layout(width="420px"))

    status_out = widgets.Output()
    summary_out = widgets.Output()
    route_rows_box = widgets.VBox([])
    route_rows_title = widgets.HTML("<b>Routes</b>")

    ui_state = {"items": [], "root": None}

    def _on_process(_):
        with status_out:
            clear_output()
            print("Scanning routes...")

        with summary_out:
            clear_output()

        ui_state["items"] = []
        route_rows_box.children = []
        proceed_btn.disabled = True

        try:
            input_path_obj = Path(path_w.value)
            files = _collect_input_files(input_path_obj)
            if not files:
                with status_out:
                    clear_output()
                    print("No .kmz/.zip/.kml files found.")
                return

            root = input_path_obj if input_path_obj.is_dir() else input_path_obj.parent
            ui_state["root"] = root
            kml_dir = root / "KML"

            scan_progress.max = len(files)
            scan_progress.value = 0

            route_widgets = []

            for idx, src in enumerate(files, start=1):
                src = Path(src)
                suffix = src.suffix.lower()

                try:
                    if suffix in {".kmz", ".zip"}:
                        kml_path = _extract_template_kml_from_zip(src, kml_dir, root, overwrite=overwrite_w.value)
                    elif suffix == ".kml":
                        kml_path = src
                    else:
                        continue
                except Exception as e:
                    status_lbl = widgets.HTML(f"<span style='color:#b00020;'>KML extract error: {e}</span>")
                    row = widgets.HBox([
                        widgets.HTML(f"<code>{src.name}</code>", layout=widgets.Layout(width="330px")),
                        widgets.HTML("<span>—</span>", layout=widgets.Layout(width="170px")),
                        widgets.HTML("<span>—</span>", layout=widgets.Layout(width="220px")),
                        status_lbl
                    ])
                    route_widgets.append(row)
                    ui_state["items"].append({
                        "src": src, "kml_path": None, "waypoints": None, "route_height_mode": None,
                        "requires_start": False, "start_w": None, "status_w": status_lbl, "row": row,
                        "scan_error": str(e),
                    })
                    scan_progress.value = idx
                    continue

                try:
                    waypoints, route_height_mode = _parse_waypoints_from_kml(kml_path, sort_by_wpml_index=sort_w.value)
                    requires_start = any(
                        str(wp.get("height_mode") or "").strip().lower() == "relativetostartpoint"
                        for wp in (waypoints or [])
                    )
                    prefill = _resolve_prefill_start_height(src, kml_path)

                    if requires_start:
                        start_w = widgets.Text(
                            value=_fmt_float_or_blank(prefill),
                            placeholder="start ellipsoidal height [m]",
                            layout=widgets.Layout(width="210px"),
                        )
                        start_cell = start_w
                    else:
                        start_w = None
                        start_cell = widgets.HTML("<span style='color:#666;'>not needed</span>",
                                                  layout=widgets.Layout(width="210px"))

                    hm_txt = route_height_mode if route_height_mode else "n/a"
                    hm_html = f"<span style='color:#b26a00;'><b>{hm_txt}</b></span>" if requires_start else f"<span>{hm_txt}</span>"
                    status_lbl = widgets.HTML("<span style='color:#666;'>Ready</span>")

                    row = widgets.HBox([
                        widgets.HTML(f"<code>{src.name}</code>", layout=widgets.Layout(width="330px")),
                        widgets.HTML(hm_html, layout=widgets.Layout(width="170px")),
                        start_cell,
                        status_lbl,
                    ])

                    route_widgets.append(row)
                    ui_state["items"].append({
                        "src": src, "kml_path": kml_path, "waypoints": waypoints,
                        "route_height_mode": route_height_mode, "requires_start": requires_start,
                        "start_w": start_w, "status_w": status_lbl, "row": row, "scan_error": None,
                    })

                except Exception as e:
                    status_lbl = widgets.HTML(f"<span style='color:#b00020;'>Parse error: {e}</span>")
                    row = widgets.HBox([
                        widgets.HTML(f"<code>{src.name}</code>", layout=widgets.Layout(width="330px")),
                        widgets.HTML("<span>—</span>", layout=widgets.Layout(width="170px")),
                        widgets.HTML("<span>—</span>", layout=widgets.Layout(width="220px")),
                        status_lbl
                    ])
                    route_widgets.append(row)
                    ui_state["items"].append({
                        "src": src, "kml_path": kml_path, "waypoints": None, "route_height_mode": None,
                        "requires_start": False, "start_w": None, "status_w": status_lbl, "row": row,
                        "scan_error": str(e),
                    })

                scan_progress.value = idx

            header = widgets.HBox([
                widgets.HTML("<b>Route file</b>", layout=widgets.Layout(width="330px")),
                widgets.HTML("<b>heightMode</b>", layout=widgets.Layout(width="170px")),
                widgets.HTML("<b>Start h (m)</b>", layout=widgets.Layout(width="220px")),
                widgets.HTML("<b>Status</b>"),
            ])
            route_rows_box.children = tuple([header] + route_widgets)
            proceed_btn.disabled = False

            n_need = sum(1 for it in ui_state["items"] if it.get("requires_start"))
            with status_out:
                clear_output()
                print(f"Scan done. {len(ui_state['items'])} route(s) found. {n_need} route(s) need start height input.")
                print("Enter missing values in the route rows, then click 'Proceed'.")

        except Exception as e:
            with status_out:
                clear_output()
                print("ERROR during scan:", e)

    def _on_proceed(_):
        with summary_out:
            clear_output()

        items = ui_state.get("items", [])
        if not items:
            with status_out:
                clear_output()
                print("Nothing to process yet. Click 'Process' first.")
            return

        try:
            sample_spacing_val = float(str(sample_dist_w.value).strip())
        except Exception:
            with status_out:
                clear_output()
                print(f"Invalid spacing value: {sample_dist_w.value}")
            return

        route_height_map = {}
        missing_routes = []
        invalid_routes = []

        for it in items:
            if it.get("scan_error") or not it.get("requires_start"):
                continue

            sw = it.get("start_w")
            raw = "" if sw is None else str(sw.value).strip()
            if raw == "":
                missing_routes.append(it["src"].name)
                continue
            try:
                val = float(raw)
            except Exception:
                invalid_routes.append((it["src"].name, raw))
                continue

            src = it["src"]
            kml_path = it["kml_path"]
            route_height_map[src.stem] = val
            route_height_map[src.name] = val
            if kml_path is not None:
                route_height_map[Path(kml_path).stem] = val
                route_height_map[Path(kml_path).name] = val

        if missing_routes or invalid_routes:
            with status_out:
                clear_output()
                print("Please fix route start heights before proceeding.")
                if missing_routes:
                    print("Missing values for:")
                    for r in missing_routes:
                        print(" -", r)
                if invalid_routes:
                    print("Invalid numeric values:")
                    for r, v in invalid_routes:
                        print(f" - {r}: {v}")
            return

        with status_out:
            clear_output()
            print("Processing routes...")

        root = ui_state["root"]
        ply_dir = root / "PLY"
        txt_dir = root / "TXT"

        proc_items = [it for it in items]
        proc_progress.max = max(1, len(proc_items))
        proc_progress.value = 0

        records = []

        for idx, it in enumerate(proc_items, start=1):
            src = it["src"]
            status_w = it["status_w"]
            status_w.value = "<span style='color:#666;'>Processing...</span>"

            if it.get("scan_error") or it.get("waypoints") is None or it.get("kml_path") is None:
                records.append({
                    "source_file": str(src), "kml_path": None if it.get("kml_path") is None else str(it["kml_path"]),
                    "ply_path": None, "txt_path": None, "n_waypoints_original": 0, "n_waypoints_output": 0,
                    "route_height_mode_detected": it.get("route_height_mode"),
                    "relativeToStartPoint_used": bool(it.get("requires_start")),
                    "start_ellipsoid_height_used": None,
                    "sampling_enabled": bool(sample_w.value),
                    "sample_spacing_m": sample_spacing_val if sample_w.value else None,
                    "target_crs": None, "best_transform_available": None,
                    "status": f"SKIPPED (scan error): {it.get('scan_error')}",
                })
                status_w.value = "<span style='color:#b00020;'>Skipped (scan error)</span>"
                proc_progress.value = idx
                continue

            waypoints = it["waypoints"]
            kml_path = it["kml_path"]

            try:
                waypoints_h, start_height_used, requires_start = _compute_actual_ellipsoidal_heights(
                    waypoints=waypoints, source_file=src, kml_path=kml_path,
                    start_heights_by_route=route_height_map, default_start_height=None, prompt_if_missing=False,
                )

                xyz, src_crs_obj, dst_crs_obj, tg = _transform_waypoints(
                    waypoints_h, target_crs=crs_w.value, height_mode="ellipsoidal", grid_dir=GRID_DIR,
                )

                waypoints_out, xyz_out = _sample_waypoints_by_distance(
                    waypoints_h, xyz, enabled=bool(sample_w.value), spacing_m=sample_spacing_val
                )

                rel_src = _safe_relative(src, root)
                out_ply = (ply_dir / rel_src).with_suffix(".ply")
                out_txt = (txt_dir / rel_src).with_suffix(".txt")

                _write_ply_ascii_with_edges(out_ply=out_ply, xyz=xyz_out, dst_crs=dst_crs_obj, overwrite=overwrite_w.value)

                txt_written = None
                txt_err = None
                if txt_w.value:
                    try:
                        _write_txt_waypoints(out_txt=out_txt, waypoints=waypoints_out, xyz=xyz_out,
                                             delimiter=TXT_DELIMITER, overwrite=overwrite_w.value)
                        txt_written = str(out_txt)
                    except Exception as e:
                        txt_err = str(e)

                status = "OK" if txt_err is None else f"PLY OK, TXT ERROR: {txt_err}"
                status_w.value = "<span style='color:#0a7f2e;'>OK</span>" if txt_err is None else "<span style='color:#b26a00;'>PLY OK, TXT issue</span>"

                records.append({
                    "source_file": str(src), "kml_path": str(kml_path), "ply_path": str(out_ply), "txt_path": txt_written,
                    "n_waypoints_original": len(waypoints_h), "n_waypoints_output": len(waypoints_out),
                    "route_height_mode_detected": it.get("route_height_mode"),
                    "relativeToStartPoint_used": requires_start,
                    "start_ellipsoid_height_used": start_height_used,
                    "sampling_enabled": bool(sample_w.value),
                    "sample_spacing_m": sample_spacing_val if sample_w.value else None,
                    "target_crs": dst_crs_obj.to_string(),
                    "best_transform_available": getattr(tg, "best_available", None),
                    "status": status,
                })

            except Exception as e:
                status_w.value = f"<span style='color:#b00020;'>Error: {e}</span>"
                records.append({
                    "source_file": str(src), "kml_path": str(kml_path), "ply_path": None, "txt_path": None,
                    "n_waypoints_original": 0 if waypoints is None else len(waypoints), "n_waypoints_output": 0,
                    "route_height_mode_detected": it.get("route_height_mode"),
                    "relativeToStartPoint_used": bool(it.get("requires_start")),
                    "start_ellipsoid_height_used": None,
                    "sampling_enabled": bool(sample_w.value),
                    "sample_spacing_m": sample_spacing_val if sample_w.value else None,
                    "target_crs": None, "best_transform_available": None,
                    "status": f"ERROR: {e}",
                })

            proc_progress.value = idx

        with status_out:
            clear_output()
            print("Done.")
            print(f"PLY output directory: {ply_dir}")
            if txt_w.value:
                print(f"TXT output directory: {txt_dir}")

        with summary_out:
            clear_output()
            if pd is not None:
                display(pd.DataFrame(records))
            else:
                for r in records:
                    print(r)

    process_btn.on_click(_on_process)
    proceed_btn.on_click(_on_proceed)

    controls_top = widgets.HBox([crs_w, sort_w, txt_w, overwrite_w])
    controls_sampling = widgets.HBox([sample_w, sample_dist_w])
    controls_buttons = widgets.HBox([process_btn, proceed_btn])
    progress_box = widgets.VBox([scan_progress, proc_progress])

    display(widgets.VBox([
        path_w,
        controls_top,
        controls_sampling,
        controls_buttons,
        progress_box,
        status_out,
        route_rows_title,
        route_rows_box,
        summary_out,
    ]))

except Exception as e:
    print("ipywidgets UI not available:", e)


In [ ]:

# =========================
# Example (sample uploaded in this ChatGPT session)
# =========================
# Uncomment to test here:
#
# sample_kmz = r"/mnt/data/KrommebrugR-533coordinateturn18wp.kmz"
# records, summary = process_kmz_kml_to_ply(
#     input_path=sample_kmz,
#     target_crs="lambert2008",
#     grid_dir=None,  # local Windows grid folder not available in this sandbox
#     start_heights_by_route={},
#     default_start_ellipsoid_height=None,
#     prompt_start_height_per_route=False,
#     export_txt=True,
# )
# display(summary)
